# Diabetis risk classifier

## Author: José Ángel de Bustos Pérez
## License: [GNU GENERAL PUBLIC LICENSE v3](https://www.gnu.org/licenses/gpl-3.0.html

A hospital network wants to train a diabetes-risk classifier across patient records that live in different clinics. Each clinic considers its records confidential. These records cannot be shared.

Homomorphic encryption allows every clinic encrypt records under a shared public key, ship the ciphertexts to a central trainer, and obtain a model without anyone (not even the trainer) is able to see patient data.

The risk factors used here are the well-known ones from epidemiology (age, BMI, blood-pressure, glucose, family history, ...).

Synthetic data is generated for the demo so real data is not being used.

## Auxiliar software

For the sake of simplicity we will create auxiliar functions/classes that will be used in this example:

1. Synthetic data generation.
2. CKKS implementation (Full Homomorphic Encryption algorithm).

### Synthetic data generation

We will create random data to train and test the algoritm. The generated data will be random and it will not be based on any real data/patterns.

The purpose of this work is illustrate how FHE can be used and not to provide an accurate diabetis-risk algorithm.

In [1]:
"""
Synthetic data for the FHE logistic regression demo.
=====================================================

1. Sample n_features columns from independent Gaussian distributions with
   arbitrary, hand-picked means and standard deviations.
2. Compute a latent score as a fixed linear combination of the features
   plus Gaussian noise. The combining coefficients are arbitrary.
3. Threshold the latent score (with optional Bernoulli sampling) to get a
   binary label.

The feature names ("age", "bmi", ...) are kept only as readable labels for
the demo output. Treat them as feature_0..feature_7 with friendlier
nicknames; their numerical ranges and weights are made up.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np

# Readable labels for the eight synthetic features. Purely cosmetic.
FEATURE_NAMES = [
    "feature_age",
    "feature_bmi",
    "feature_glucose",
    "feature_bp",
    "feature_hba1c",
    "feature_family",
    "feature_activity",
    "feature_waist",
]

# Arbitrary marginal distributions (mean, std) for each feature.
# These were picked by hand so the feature values look plausible when
# printed; they are NOT calibrated to any real-world dataset.
FEATURE_MARGINALS = [
    (50.0, 15.0),     # feature_age
    (26.0, 5.0),      # feature_bmi
    (95.0, 15.0),     # feature_glucose
    (125.0, 15.0),    # feature_bp
    (5.5, 0.8),       # feature_hba1c
    (0.0, 1.0),       # feature_family  (will be binarised below)
    (4.0, 3.0),       # feature_activity
    (90.0, 12.0),     # feature_waist
]

# Arbitrary coefficients for the latent label-generating function.
# These determine how strongly each feature pushes the label towards 1.
# They are NOT estimates from any study.
TRUE_COEFFS = np.array([
    0.5,   # feature_age
    0.8,   # feature_bmi
    1.0,   # feature_glucose
    0.3,   # feature_bp
    1.1,   # feature_hba1c
    0.5,   # feature_family
    -0.4,  # feature_activity   (negative => protective)
    0.6,   # feature_waist
])

@dataclass
class SyntheticDataset:
    """A synthetic binary-classification dataset."""
    X: np.ndarray                    # shape (n, n_features)
    y: np.ndarray                    # shape (n,), 0/1
    feature_names: List[str] = field(default_factory=lambda: list(FEATURE_NAMES))

    def describe(self) -> str:
        lines = [f"SyntheticDataset: {self.X.shape[0]} rows, "
                 f"{self.X.shape[1]} features"]
        lines.append(f"  positive rate (y=1): {self.y.mean():.2%}")
        lines.append("  feature                  mean      std")
        for i, name in enumerate(self.feature_names):
            col = self.X[:, i]
            lines.append(f"  {name:<22} {col.mean():8.2f}  {col.std():7.2f}")
        return "\n".join(lines)

# Create a random synthetic dataset for binary classification.
def generate_dataset(
    n_samples: int = 600, # Number of patients
    noise_std: float = 0.5, # Standard deviation of additive Gaussian noise on the latent score.
                            # Larger values make the problem harder.
    seed: int = 7, # Seed for reproducibility
) -> SyntheticDataset:

    rng = np.random.default_rng(seed)
    n_features = len(FEATURE_MARGINALS)

    # Sample features independently from the chosen marginals (normal distribution)
    X = np.empty((n_samples, n_features), dtype=np.float64)
    for j, (mu, sd) in enumerate(FEATURE_MARGINALS):
        X[:, j] = rng.normal(loc=mu, scale=sd, size=n_samples)

    # Binarise the "family" column to 0/1 so we have a categorical feature
    # in the mix. The 0.9 threshold is arbitrary.
    # Values bigger than threshold -> True
    fam_idx = FEATURE_NAMES.index("feature_family")
    X[:, fam_idx] = (X[:, fam_idx] > 0.9).astype(float) 

    # Latent score: standardise the columns, weight them, add noise.
    # Standardising means the coefficients in TRUE_COEFFS operate on a
    # comparable scale per feature
    Xz = (X - X.mean(axis=0)) / X.std(axis=0)
    latent = Xz @ TRUE_COEFFS + rng.normal(0, noise_std, n_samples)

    # Bernoulli draw via a logistic link. The intercept is chosen so the
    # positive rate is roughly balanced
    intercept = -np.median(latent)
    probs = 1.0 / (1.0 + np.exp(-(latent + intercept)))
    y = rng.binomial(1, probs).astype(int)

    return SyntheticDataset(X=X, y=y)

# Split dataset into Train and Test data
def split(
    ds: SyntheticDataset,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> Tuple[SyntheticDataset, SyntheticDataset]:
    rng = np.random.default_rng(seed)
    n = ds.X.shape[0]
    idx = rng.permutation(n)
    n_test = int(round(n * test_fraction))
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    train = SyntheticDataset(ds.X[train_idx], ds.y[train_idx], ds.feature_names)
    test = SyntheticDataset(ds.X[test_idx], ds.y[test_idx], ds.feature_names)
    
    return train, test

### CCKS algorithm with bootstrapping

We will use the OpenFHE library to implement the CKKS algorithm.

Bootstrapping is enabled so that the ciphertext **level budget** is refreshed between epochs and gradient-descent can run for arbitrarily many iterations.

In [2]:
from __future__ import annotations

import math
from typing import List, Tuple, Optional

def _import_openfhe():
    global CCParamsCKKSRNS, GenCryptoContext, PKESchemeFeature
    global SecurityLevel, SecretKeyDist, ScalingTechnique
    from openfhe import (  
        CCParamsCKKSRNS,
        GenCryptoContext,
        PKESchemeFeature,
        SecurityLevel,
        SecretKeyDist,
        ScalingTechnique,
    )

## FHE

In [3]:
"""
End-to-end demo
===============

Scenario (motivation only)
--------------------------
A hospital network wants to train a diabetes-risk classifier across patient
records that live in different clinics. Patient data is PHI and cannot
leave the clinics in the clear. Each clinic encrypts its records under a
shared CKKS public key; a central trainer fits a logistic-regression model
on the ciphertexts; the trained model itself stays encrypted and is only
decrypted by the clinics when they want to score a new patient.

This is the motivating *use case*. The actual data used by this script is
purely synthetic — features and weights are arbitrary random values, not
calibrated to any clinical study. See `synthetic_data.py`.

This script simulates the workflow on a single machine:
  1. Generate a synthetic dataset.
  2. Standardise features (a clinic-side preprocessing step).
  3. Encrypt the training data, train logistic regression under FHE.
  4. Encrypt a held-out test set, run inference under FHE.
  5. Decrypt predictions and report accuracy + a few example scores.

Run with:
    python demo.py
"""

import numpy as np

#from fhe_logreg import FHELogisticRegression, CKKSConfig


def standardize_train_test(X_train, X_test):
    """Mean-zero unit-variance scaling, fit on train only.

    Why: the polynomial sigmoid used inside FHE is accurate only on
    roughly [-8, 8]. Standardising the features keeps z = w·x in range.
    """
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    sd[sd == 0] = 1.0
    return (X_train - mu) / sd, (X_test - mu) / sd


def report(name, y_true, y_pred):
    acc = (y_pred == y_true).mean()
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    print(f"[{name}] acc={acc:.3f}  sensitivity={sens:.3f}  "
          f"specificity={spec:.3f}  (TP={tp}, TN={tn}, FP={fp}, FN={fn})")

We will create a synthetic random sample:

In [4]:
import random

# Generate a random number of patients between 400 and 1000
n_patients = random.randint(400, 1000)

cohort = generate_dataset(n_samples=n_patients, seed=7)
print(cohort.describe())

SyntheticDataset: 511 rows, 8 features
  positive rate (y=1): 52.25%
  feature                  mean      std
  feature_age               47.99    14.05
  feature_bmi               25.94     4.72
  feature_glucose           95.09    15.92
  feature_bp               124.64    14.79
  feature_hba1c              5.50     0.80
  feature_family             0.18     0.39
  feature_activity           4.11     2.85
  feature_waist             90.17    12.09


We split the data into **train** (75%) and **test** (25%) datasets:

In [5]:
train, test = split(cohort, test_fraction=0.25, seed=0)
X_train, X_test = standardize_train_test(train.X, test.X)

print(f"X_train size:  {X_train.shape}")
print(f"X_test size: {X_test.shape}")

X_train size:  (383, 8)
X_test size: (128, 8)


Training using Full Homomorphic Encryption using CKKS algorithm:

In [6]:
cfg = CKKSConfig(
    ring_dim=1 << 16,        # 65536 — needed for 128-bit secure bootstrap
    batch_size=1 << 15,      # 32768 SIMD slots
    mult_depth=25,
    levels_after_bootstrap=10,
)
    
clf = FHELogisticRegression(
    cfg=cfg,
    learning_rate=1.0,
    epochs=20,
    bootstrap_every=2,       # one bootstrap every two epochs
)

print(">>> Encrypting training data and running gradient descent under FHE...")
clf.fit(X_train, train.y)

NameError: name 'CKKSConfig' is not defined

In [ ]:






def main():
    # ---- 1. Synthetic cohort ------------------------------------------------
    cohort = generate_dataset(n_samples=600, seed=7)
    print(cohort.describe())
    print()

    train, test = split(cohort, test_fraction=0.25, seed=0)
    X_train, X_test = standardize_train_test(train.X, test.X)

    # ---- 2. Train under FHE -------------------------------------------------
    cfg = CKKSConfig(
        ring_dim=1 << 16,        # 65536 — needed for 128-bit secure bootstrap
        batch_size=1 << 15,      # 32768 SIMD slots
        mult_depth=25,
        levels_after_bootstrap=10,
    )
    clf = FHELogisticRegression(
        cfg=cfg,
        learning_rate=1.0,
        epochs=20,
        bootstrap_every=2,       # one bootstrap every two epochs
    )

    print(">>> Encrypting training data and running gradient descent under FHE...")
    clf.fit(X_train, train.y)

    # ---- 3. Inspect learned model ------------------------------------------
    w = clf.decrypt_weights()
    print("\nLearned coefficients (intercept first):")
    print(f"  intercept              {w[0]:+.3f}")
    for name, coef in zip(cohort.feature_names, w[1:]):
        print(f"  {name:<22} {coef:+.3f}")
    print(
        "  (positive = increases risk, negative = protective. "
        "Coefficients are on the standardised scale.)"
    )

    # ---- 4. Inference under FHE on the test set -----------------------------
    print("\n>>> Encrypting test cohort and computing risk scores under FHE...")
    enc_preds = clf.predict_encrypted(X_test)

    # The clinic (key holder) decrypts.
    pt = clf.cc.Decrypt(enc_preds, clf.keys.secretKey)
    pt.SetLength(X_test.shape[0])
    risk_scores = np.array(pt.GetRealPackedValue())
    preds = (risk_scores >= 0.5).astype(int)

    # ---- 5. Report ----------------------------------------------------------
    print()
    report("FHE-LR", test.y, preds)

    print("\nExample patients from the test set:")
    print(f"  {'idx':>3}  {'risk score':>10}  {'pred':>4}  {'truth':>5}")
    for i in range(min(10, len(preds))):
        print(f"  {i:>3}  {risk_scores[i]:>10.3f}  "
              f"{int(preds[i]):>4}  {int(test.y[i]):>5}")


if __name__ == "__main__":
    main()